[Attention is all you need!](https://arxiv.org/pdf/1706.03762)

[Transformers, the tech behind LLMs | Deep Learning Chapter 5](https://www.youtube.com/watch?v=wjZofJX0v4M)

## Modelowanie Języka z Wykorzystaniem Sieci Transformatorowych

Laboratoria skupiają się na przełożeniu teorii na działający kod, budując od zera architekturę zaproponowaną w pracy *"Attention Is All You Need"*. Naszym celem jest zrozumienie mechaniki działania modeli pozbawionych warstw rekurencyjnych i konwolucyjnych, bazujących wyłącznie na mechanizmach uwagi.

<center><img src="https://media.geeksforgeeks.org/wp-content/uploads/20251004125140134507/transformers.webp" width=70%><br><i>www.geeksforgeeks.org</i></center>

### Etapy Realizacji Projektu

Pracę nad modelem zrealizujemy w trzech głównych fazach, łącząc poszczególne operacje tensorowe w coraz większe moduły:



#### **Faza I: Reprezentacja i Pozycjonowanie**

W pierwszym kroku musimy przetworzyć dyskretne tokeny wejściowe (w postaci macierzy o wymiarach `[wielkość_partii, długość_sekwencji]`) na ciągłą przestrzeń wektorową. Zaimplementujemy warstwę rzutowań (*Embeddings*), która rozszerzy nasz tensor o dodatkowy wymiar modelu (w oryginalnej pracy przyjęto $d_{model}=512$ ). Ponieważ nasza sieć nie przetwarza danych sekwencyjnie, w tym miejscu konieczne będzie również dodanie sygnału o pozycji każdego słowa za pomocą funkcji trygonometrycznych (kodowanie pozycyjne).




#### **Faza II: Inżynieria Mechanizmu Uwagi**

Sercem naszego modelu będzie moduł *Scaled Dot-Product Attention*. Przy jego implementacji kluczowe będzie zwrócenie uwagi na prawidłowe skalowanie wyników przez odwrotność pierwiastka z wymiaru klucza ($\frac{1}{\sqrt{d_k}}$)  – pominięcie tego kroku drastycznie pogarsza stabilność uczenia.

Następnie obudujemy ten mechanizm w architekturę wielogłową (*Multi-Head Attention*), co wymusi na nas operowanie na tensorach o kształcie `[wielkość_partii, liczba_głów, długość_sekwencji, długość_sekwencji]`. W tej fazie przygotujemy również logikę maskowania wartości. Maska przyczynowa w dekoderze będzie kluczowa, aby zablokować modelowi dostęp do przyszłych kontekstów, co zrealizujemy poprzez ustawienie odpowiednich wartości na $-\infty$ tuż przed aplikacją funkcji softmax.

#### **Faza III: Składanie Warstw Ukrytych i Kompozycja Modelu**

Gdy moduły uwagi będą gotowe, połączymy je z w pełni połączonymi sieciami jednokierunkowymi (*Feed-Forward Networks*). Warto pamiętać, że wewnętrzna warstwa ukryta tych sieci znacząco rozszerza wymiarowość (standardowo do $d_{ff}=2048$ ).

Każda podwarstwa zostanie otoczona mechanizmem normalizacji (*Layer Normalization*) oraz połączeniem omijającym (rezydualnym). Na tym etapie należy bezwzględnie pilnować zgodności kształtów tensorów na wejściu i wyjściu poszczególnych bloków. Gotowe bloki ułożymy w stosy kodera i dekodera (po 6 takich warstw ), a proces zakończymy transformacją liniową i nałożeniem funkcji softmax w celu wygenerowania ostatecznych prawdopodobieństw dla słownika.


### Weryfikacja i Optymalizacja Złożoności

Podczas pisania kodu zachęcam do testowania każdej funkcji z osobna, upewniając się, że gradienty przepływają swobodnie, a maski faktycznie zerują niepożądane połączenia. Należy mieć na uwadze, że samo-uwaga charakteryzuje się kwadratową złożonością pamięciową i obliczeniową względem długości analizowanego tekstu ($O(n^2 \cdot d)$). Będzie to wymagało od Was rozważnego dobierania rozmiaru paczki danych (batch size) podczas późniejszych prób treningowych, aby nie przekroczyć dostępnej pamięci VRAM. Nie zapomnijcie również o zaimplementowaniu mechanizmu odrzucania (*Dropout*)  jako formy regularyzacji.

### Zadanie 1: Zamiana słów na wektory (Embeddings) i Informacja o Pozycji

W tym zadaniu zbudujecie moduł wejściowy sieci. Transformery różnią się od starszych sieci (rekurencyjnych) tym, że nie przetwarzają tekstu słowo po słowie, tylko widzą całe zdanie od razu. Z tego powodu musimy przekazać modelowi nie tylko to, co oznaczają poszczególne słowa, ale też na którym miejscu w zdaniu się one znajdują.

**Wskazówki do napisania kodu:**

* **Zamiana tokenów na wektory (Embeddings) -** Użyjcie gotowego modułu `nn.Embedding` z biblioteki PyTorch, żeby zamienić słowa na gęste wektory. Załóżcie, że początkowy rozmiar wektora słowa może być czasem inny niż główny rozmiar modelu (wtedy trzeba użyć dodatkowej warstwy liniowej, by je zrównać). Na samym początku funkcji `forward` dodajcie zabezpieczenie (asercję), które sprawdzi, czy dane wejściowe mają poprawny typ – w tym przypadku powinien to być `torch.long`.

* **Skalowanie wektorów -** Ważnym krokiem opisanym w artykule jest pomnożenie wektorów słów przez pierwiastek kwadratowy z głównego rozmiaru modelu (czyli $\sqrt{d_{model}}$). Żeby kod działał szybciej, policzcie ten pierwiastek tylko raz i zapiszcie go jako ułamek (`float`) już w konstruktorze `__init__`.

* **Dodanie informacji o pozycji słowa -** Żeby model wiedział, w jakiej kolejności ułożone są słowa w zdaniu, stwórzcie kodowanie pozycji oparte na funkcjach trygonometrycznych – sinusach i kosinusach. Ponieważ informacje o pozycji oraz wektory słów muszą mieć ten sam rozmiar, po prostu dodajemy je do siebie. Pamiętajcie tylko o dopasowaniu kształtu tensora pozycji – trzeba go "rozszerzyć" o wymiar paczki danych (*batch dimension*), żeby dodawanie zadziałało bez błędów.

* **Zabezpieczenie przed przeuczeniem (Dropout) -** Na samym końcu, po dodaniu do siebie wektorów słów i wartości pozycji, przepuśćcie ten wynik przez warstwę *Dropout*.


In [1]:
import torch
import torch.nn as nn
import math

class PositionalSignal(nn.Module):
    """
    Moduł dodający kodowanie pozycyjne do embeddingów.
    Tworzy macierz pozycji raz w __init__, a potem tylko ją przycina w forward().
    """
    def __init__(self, d_model, max_len=5000):
        super().__init__()

        # TODO (student)
        # 1) Utwórz tensor na kodowanie pozycyjne o kształcie [max_len, d_model]
        #    Użyj: torch.zeros(...)
        pe = torch.zeros(max_len, d_model)

        # 2) Utwórz indeksy pozycji [0, 1, 2, ... max_len-1]
        #    Kształt ma być [max_len, 1], dlatego dodaj wymiar przez .unsqueeze(1)
        #    Użyj: torch.arange(..., dtype=torch.float).unsqueeze(1)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # 3) Oblicz współczynniki skalujące dla parzystych wymiarów embeddingu
        #    (częstotliwości dla sin/cos)
        #    Użyj: torch.arange(0, d_model, 2), torch.exp(...), math.log(...)
        #    Wynik powinien mieć kształt [d_model // 2]
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float) * (-math.log(10000.0) / d_model))

        # 4) Wpisz sinusy do parzystych kolumn (0, 2, 4, ...)
        #    Użyj indeksowania pe[:, 0::2] oraz torch.sin(...)
        pe[:, 0::2] = torch.sin(position * div_term)

        # 5) Wpisz cosinusy do nieparzystych kolumn (1, 3, 5, ...)
        #    Użyj indeksowania pe[:, 1::2] oraz torch.cos(...)
        pe[:, 1::2] = torch.cos(position * div_term)

        # 6) Dodaj wymiar batcha z przodu, aby uzyskać kształt [1, max_len, d_model]
        #    Użyj: pe.unsqueeze(0)
        #
        # 7) Zarejestruj tensor jako buffer:
        #    - nie jest trenowany (to nie są wagi),
        #    - ale zapisuje się razem z modelem,
        #    - i przenosi się poprawnie na CPU/GPU.
        #    Użyj: self.register_buffer("nazwa", tensor)
        self.register_buffer("pe_matrix", pe.unsqueeze(0))

    def forward(self, x):
        # x ma zwykle kształt: [batch_size, seq_len, d_model]
        # Pobieramy tylko tyle pozycji, ile wynosi długość aktualnej sekwencji (seq_len)
        # x.size(1) = seq_len
        #
        # self.pe_matrix[:, :x.size(1)] ma kształt [1, seq_len, d_model]
        # PyTorch zastosuje broadcasting po wymiarze batch_size przy dodawaniu.
        return x + self.pe_matrix[:, :x.size(1)]


class TransformerInput(nn.Module):
    """
    Moduł wejściowy transformera:
    tokeny -> embeddingi -> (opcjonalna projekcja) -> dodanie pozycji -> dropout
    """
    def __init__(self, vocab_size, d_model, d_embed, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model

        # TODO (student)
        # 1) Warstwa embedding:
        #    zamienia indeksy tokenów (int) na wektory o rozmiarze d_embed
        #    Użyj: nn.Embedding(vocab_size, d_embed)
        self.word_lookup = nn.Embedding(vocab_size, d_embed)

        # 2) Jeśli d_embed != d_model, trzeba dopasować wymiar embeddingu do wymiaru modelu.
        #    - gdy różne: użyj nn.Linear(d_embed, d_model)
        #    - gdy takie same: użyj nn.Identity() (nic nie zmienia)
        self.resizer = nn.Linear(d_embed, d_model) if d_embed != d_model else nn.Identity()

        # 3) Zapisz skalę sqrt(d_model) (jako liczbę), aby nie liczyć jej w każdej iteracji forward()
        #    Użyj: math.sqrt(d_model)
        self.embed_scale = math.sqrt(d_model)

        # 4) Utwórz moduł pozycyjny i dropout
        #    Użyj: PositionalSignal(d_model), nn.Dropout(...)
        self.pos_module = PositionalSignal(d_model)
        self.regularization = nn.Dropout(dropout_rate)

    def forward(self, tokens):
        # tokens powinno mieć dtype=torch.long, bo nn.Embedding przyjmuje indeksy całkowite
        if not tokens.dtype == torch.long:
            raise TypeError(f"Oczekiwano torch.long, otrzymano {tokens.dtype}")

        # TODO (student)
        # 1) Zamień tokeny na embeddingi:
        #    [batch, seq_len] -> [batch, seq_len, d_embed]
        #    Użyj: self.word_lookup(tokens)
        #
        # 2) Przeskaluj embeddingi przez sqrt(d_model)
        #    (mnożenie przez self.embed_scale)
        x = self.word_lookup(tokens) * self.embed_scale

        # 3) Dopasuj wymiar do d_model (jeśli trzeba)
        #    [batch, seq_len, d_embed] -> [batch, seq_len, d_model]
        x = self.resizer(x)

        # 4) Dodaj kodowanie pozycyjne (ten sam kształt co x)
        #    Użyj: self.pos_module(x)
        x = self.pos_module(x)

        # 5) Zastosuj dropout i zwróć wynik
        #    To jest gotowe wejście do kolejnych bloków transformera (np. enkodera)
        return self.regularization(x)

### Zadanie 2: Mechanizm Wielogłowej Uwagi (Multi-Head Attention)

W tym etapie zaimplementujecie serce modelu Transformer. Mechanizm ten pozwala sieci skupić się na różnych fragmentach zdania jednocześnie, co opisano w sekcjach 3.2.1 oraz 3.2.2 artykułu. Zamiast liczyć jedną "uśrednioną" uwagę, model dzieli dane na wiele "głów", z których każda może uczyć się innych zależności językowych.

Kluczowe zasady mechanizmu:

* **Skalowany Iloczyn Skalarny (Scaled Dot-Product)** - Podstawą jest wzór $Attention(Q,K,V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$. Dzielenie przez $\sqrt{d_k}$ zapobiega powstawaniu zbyt dużych wartości, które mogłyby "zamrozić" proces uczenia się (małe gradienty w funkcji softmax).

* **Wielogłowość (Multi-Head)** - Zamiast jednej operacji na dużym wektorze, rzutujemy zapytania (Q), klucze (K) i wartości (V) $h$ razy do mniejszych wymiarów. W pracy przyjęto $h=8$ głowic, z których każda operuje na wymiarze $d_k = 64$.
* **Maskowanie** - W dekoderze musimy zablokować możliwość "zaglądania w przyszłość". Robimy to, dodając do wyników przed softmaxem bardzo małą liczbę ($-\infty$), co po nałożeniu funkcji softmax wyzeruje te połączenia.

* **Uniwersalność** - Mechanizm musi obsługiwać zarówno samo-uwagę (self-attention), jak i uwagę krzyżową (cross-attention), gdzie Q pochodzi z dekodera, a K i V z enkodera.

In [2]:
import torch
import torch.nn as nn
import math

class MultiHeadAttentionModule(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        # Liczba głowic musi dzielić wymiar modelu bez reszty
        assert d_model % num_heads == 0, "d_model musi dzielić się przez num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Wymiar jednej głowy

        # TODO (student)
        # 1) Utwórz 3 warstwy liniowe do projekcji:
        #    - query (Q)
        #    - key   (K)
        #    - value (V)
        #    Każda mapuje z d_model -> d_model
        #    Użyj: nn.Linear(d_model, d_model)
        self.q_linear = nn.Linear(d_model, d_model)
        self.k_linear = nn.Linear(d_model, d_model)
        self.v_linear = nn.Linear(d_model, d_model)

        # 2) Utwórz projekcję końcową po scaleniu wszystkich głowic
        #    Również: d_model -> d_model
        self.out_projection = nn.Linear(d_model, d_model)

        # 3) Dropout na wagach uwagi (po softmax)
        self.attention_dropout = nn.Dropout(dropout)

    def split_heads(self, x, batch_size):
        """
        Rozdziela reprezentację na wiele głów.

        Wejście:
            x: [batch, seq_len, d_model]
        Wyjście:
            [batch, num_heads, seq_len, d_k]
        """
        # Krok 1: rozbij ostatni wymiar d_model na [num_heads, d_k]
        #         użyj .view(batch_size, seq_len, num_heads, d_k)
        # Krok 2: zamień kolejność wymiarów, aby num_heads było przed seq_len
        #         użyj .transpose(1, 2)
        return x.view(batch_size, -1, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        # query: [batch, seq_len_q, d_model]
        # key:   [batch, seq_len_k, d_model]
        # value: [batch, seq_len_k, d_model]
        # Uwaga: seq_len_q i seq_len_k mogą być różne (cross-attention)
        batch_size = query.size(0)

        # TODO (student)
        # 1) Przepuść query/key/value przez odpowiednie warstwy liniowe
        #    i rozdziel na głowy funkcją split_heads(...)
        #
        # Po tym kroku:
        # q, k, v -> [batch, num_heads, seq_len, d_k]
        q = self.split_heads(self.q_linear(query), batch_size)
        k = self.split_heads(self.k_linear(key), batch_size)
        v = self.split_heads(self.v_linear(value), batch_size)

        # 2) Oblicz "scores" = QK^T / sqrt(d_k)
        #    Użyj:
        #    - k.transpose(-2, -1), aby zamienić [seq_len_k, d_k] -> [d_k, seq_len_k]
        #    - torch.matmul(q, ...)
        #    - podziel przez math.sqrt(self.d_k)
        #
        # Wynik:
        # scores -> [batch, num_heads, seq_len_q, seq_len_k]
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.d_k)

        # 3) Jeśli jest maska, zablokuj wybrane pozycje przed softmax
        #    Użyj: masked_fill(mask == 0, -1e9)
        #    Dzięki temu softmax da ~0 na zablokowanych pozycjach.
        #
        # WAŻNE: maska musi dać się rozgłosić (broadcast) do kształtu scores.
        # Typowe kształty maski:
        # - [batch, 1, 1, seq_len_k]      (padding mask)
        # - [batch, 1, seq_len_q, seq_len_k]
        if mask is not None:
            scores = scores.masked_fill(mask == 0, -1e9)

        # 4) Zamień scores na wagi uwagi:
        #    - softmax po ostatnim wymiarze (po kluczach)
        #    - attention_dropout na wagach
        #    Użyj: torch.softmax(scores, dim=-1)
        weights = torch.softmax(scores, dim=-1)
        self.attention_weights = weights.detach()
        weights = self.attention_dropout(weights)

        # 5) Policz ważoną sumę wartości V:
        #    context = weights @ v
        #
        # Kształt po matmul:
        # context -> [batch, num_heads, seq_len_q, d_k]
        context = torch.matmul(weights, v)

        # 6) Scal głowy z powrotem do jednego wektora:
        #    a) transpose(1, 2): [batch, seq_len_q, num_heads, d_k]
        #    b) contiguous(): ważne przed view po transpose
        #    c) view(..., d_model): bo num_heads * d_k = d_model
        #
        # Wynik:
        # context -> [batch, seq_len_q, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        # 7) Finalna projekcja wyjściowa
        #    Użyj: self.out_projection(context)
        return self.out_projection(context)

### Zadanie 3: Pozycyjna Sieć Feed-Forward
Zgodnie z opisem w sekcji 3.3 oryginału, sieć ta jest stosowana do każdej pozycji w sekwencji całkowicie niezależnie i identycznie. Oznacza to, że każde słowo (lub token) przechodzi przez dokładnie te same operacje liniowe, co pozwala na łatwą równoległość obliczeń.

Kluczowe parametry techniczne:
- **Architektura** - Składa się z dwóch przekształceń liniowych, pomiędzy którymi znajduje się funkcja aktywacji ReLU.
- **Wymiarowość** - Warstwa wejściowa i wyjściowa mają rozmiar $d_{model} = 512$, natomiast wewnętrzna warstwa ukryta jest znacznie szersza i wynosi $d_{ff} = 2048$.
- **Równanie** - Proces ten można zapisać jako $FFN(x) = max(0, xW_1 + b_1)W_2 + b_2$.

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PositionWiseFFN(nn.Module):
    """
    Dwuwarstwowa sieć Feed-Forward stosowana niezależnie na każdej pozycji sekwencji.

    Działa na ostatnim wymiarze tensora:
    [batch, seq_len, d_model] -> [batch, seq_len, d_ff] -> [batch, seq_len, d_model]
    """
    def __init__(self, d_model, d_ff, dropout_rate=0.1):
        super().__init__()
        self.d_model = d_model
        self.d_ff = d_ff

        # TODO (student) — inicjalizacja warstw
        # 1) Pierwsza warstwa liniowa rozszerza wymiar reprezentacji:
        #    d_model -> d_ff
        #    Użyj: nn.Linear(d_model, d_ff)
        self.w_expand = nn.Linear(d_model, d_ff)

        # 2) Druga warstwa liniowa zwęża reprezentację z powrotem:
        #    d_ff -> d_model
        #    Użyj: nn.Linear(d_ff, d_model)
        self.w_shrink = nn.Linear(d_ff, d_model)

        # 3) (Opcjonalnie) zainicjalizuj wagi, np. metodą Xavier uniform,
        #    aby trening był stabilniejszy na starcie.
        #    Użyj: nn.init.xavier_uniform_(...)
        nn.init.xavier_uniform_(self.w_expand.weight)
        nn.init.xavier_uniform_(self.w_shrink.weight)

        # 4) Dropout po aktywacji (na warstwie ukrytej)
        #    Użyj: nn.Dropout(dropout_rate)
        self.dropout = nn.Dropout(dropout_rate)

    def forward(self, x):
        # x powinien mieć kształt [batch, seq_len, d_model]
        # Sprawdzamy tylko ostatni wymiar, bo FFN działa "po cechach"
        if x.size(-1) != self.d_model:
            raise ValueError(
                f"Błąd: oczekiwano ostatniego wymiaru = d_model ({self.d_model}), "
                f"otrzymano {x.size(-1)}"
            )

        # TODO (student) — przepływ danych
        # Krok 1) Rozszerzenie wymiaru + aktywacja nieliniowa
        #         Użyj: self.w_expand(x), a potem F.relu(...)
        # Wynik: [batch, seq_len, d_ff]
        hidden_state = F.relu(self.w_expand(x))

        # Krok 2) Dropout na reprezentacji ukrytej
        #         Użyj: self.dropout(...)
        hidden_state = self.dropout(hidden_state)

        # Krok 3) Projekcja z powrotem do d_model
        #         Użyj: self.w_shrink(...)
        # Wynik: [batch, seq_len, d_model]
        output = self.w_shrink(hidden_state)

        return output

In [4]:
net = PositionWiseFFN(d_model = 512,  d_ff =2048)
print(net)

PositionWiseFFN(
  (w_expand): Linear(in_features=512, out_features=2048, bias=True)
  (w_shrink): Linear(in_features=2048, out_features=512, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


### Zadanie 4: Kompletna Warstwa Enkodera

Warstwa enkodera składa się z dwóch głównych podwarstw: mechanizmu Multi-Head Self-Attention oraz sieci Position-wise Feed-Forward. Kluczowym aspektem tej konstrukcji jest to, że każda z tych podwarstw jest otoczona połączeniem rezydualnym, po którym następuje normalizacja warstwowa (Layer Normalization).

Zasady konstrukcji (Sekcja 3.1):
- **Struktura podwarstwy** - Wyjście każdej podwarstwy definiuje równanie $\text{LayerNorm}(x + \text{Sublayer}(x))$, gdzie $\text{Sublayer}(x)$ to funkcja realizowana przez dany moduł (uwaga lub FFN).
- **Regularyzacja** - Zgodnie z Sekcją 5.4, dropout należy zaaplikować do wyjścia każdej podwarstwy, zanim zostanie ono dodane do wejścia podwarstwy i znormalizowane.
- **Spójność wymiarów** - Wszystkie podwarstwy w modelu, a także warstwy osadzeń, generują wyjścia o wymiarze $d_{model} = 512$, co umożliwia działanie połączeń rezydualnych.

In [5]:
import torch
import torch.nn as nn

class EncoderBlock(nn.Module):
    """
    Pojedyncza warstwa kodera (Sekcja 3.1).
    Łączy mechanizm Self-Attention, FFN oraz Residual Connections.
    """
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        # TODO Inicjalizacja modułów
        # 1. Główny mechanizm samo-uwagi (Self-Attention)
        self.self_attention_block = MultiHeadAttentionModule(d_model, num_heads, dropout)

        # 2. Sieć Feed-Forward (FFN)
        self.feed_forward_block = PositionWiseFFN(d_model, d_ff, dropout)

        # 3. Warstwy normalizacji dla obu podwarstw (LayerNorm)
        self.norm_attention = nn.LayerNorm(d_model)
        self.norm_ffn = nn.LayerNorm(d_model)

        # 4. Dropout stosowany przed dodaniem do połączenia rezydualnego
        self.dropout_layer = nn.Dropout(dropout)

    def forward(self, x, padding_mask=None):
        """
        Realizacja struktury: LayerNorm(x + Dropout(Sublayer(x)))
        """

        # TODO Pierwsza podwarstwa - Attention
        # Krok 1: Obliczenie samo-uwagi (Query, Key i Value to ten sam tensor x)
        attn_out = self.self_attention_block(x, x, x, padding_mask)

        # Krok 2: Dropout i połączenie rezydualne z normalizacją
        x = self.norm_attention(x + self.dropout_layer(attn_out))

        # TODO Druga podwarstwa - FFN
        # Krok 3: Przetworzenie przez sieć Feed-Forward
        ffn_out = self.feed_forward_block(x)

        # Krok 4: Dropout i drugie połączenie rezydualne z normalizacją
        x = self.norm_ffn(x + self.dropout_layer(ffn_out))

        return x

In [6]:
net = EncoderBlock( d_model = 512, d_ff =2048, num_heads=8, dropout=0.1)
print(net)

EncoderBlock(
  (self_attention_block): MultiHeadAttentionModule(
    (q_linear): Linear(in_features=512, out_features=512, bias=True)
    (k_linear): Linear(in_features=512, out_features=512, bias=True)
    (v_linear): Linear(in_features=512, out_features=512, bias=True)
    (out_projection): Linear(in_features=512, out_features=512, bias=True)
    (attention_dropout): Dropout(p=0.1, inplace=False)
  )
  (feed_forward_block): PositionWiseFFN(
    (w_expand): Linear(in_features=512, out_features=2048, bias=True)
    (w_shrink): Linear(in_features=2048, out_features=512, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (norm_attention): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (norm_ffn): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout_layer): Dropout(p=0.1, inplace=False)
)


### Zadanie 5: Warstwa Dekodera i Mechanizm Maskowania

Zgodnie z Sekcją 3.1, dekoder składa się z trzech podwarstw, z których każda otoczona jest połączeniem rezydualnym i normalizacją warstwową:

- **Masked Self-Attention** - Zapobiega "zaglądaniu w przyszłość" podczas generowania tekstu.
- **Cross-Attention** - Pozwala dekoderowi skupić się na odpowiednich fragmentach zdania wejściowego (pochodzącego z enkodera).
- **Feed-Forward Network (FFN)** - Przetwarza cechy dla każdej pozycji.

Kluczowe szczegóły implementacyjne:
- **Causal Masking (Maska przyczynowa)** - Wykorzystujemy macierz trójkątną górną, aby wypełnić niedozwolone pozycje wartością $-\infty$ przed funkcją softmax. Dzięki temu przewidywanie dla pozycji $i$ zależy tylko od znanych wyjść na pozycjach mniejszych niż $i$.
- **Regularyzacja** - Podobnie jak w enkoderze, dropout stosujemy do wyjścia każdej podwarstwy przed dodaniem go do wejścia i normalizacją.

In [7]:
import torch
import torch.nn as nn

class TransformerDecoderLayer(nn.Module):
    """
    Pojedynczy blok dekodera:
    1) Masked Self-Attention   (dekoder patrzy tylko w lewo)
    2) Cross-Attention         (dekoder patrzy na wyjście enkodera)
    3) Feed-Forward Network    (FFN)

    Każda podwarstwa jest opakowana przez:
    residual connection + dropout + LayerNorm
    """
    def __init__(self, d_model, d_ff, num_heads, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # TODO (student) — inicjalizacja modułów
        # 1) Dwa mechanizmy uwagi:
        #    - self_attn: uwaga wewnątrz dekodera (masked self-attention)
        #    - cross_attn: uwaga dekodera do pamięci enkodera (encoder-decoder attention)
        #    Użyj: MultiHeadAttentionModule(d_model, num_heads, dropout)
        self.self_attn = MultiHeadAttentionModule(d_model, num_heads, dropout)
        self.cross_attn = MultiHeadAttentionModule(d_model, num_heads, dropout)

        # 2) Sieć Feed-Forward działająca na każdej pozycji niezależnie
        #    Użyj: PositionWiseFFN(d_model, d_ff, dropout)
        self.ffn_block = PositionWiseFFN(d_model, d_ff, dropout)

        # 3) Trzy osobne LayerNorm — po jednej na każdą podwarstwę
        #    Użyj: nn.LayerNorm(d_model)
        self.norm_self = nn.LayerNorm(d_model)
        self.norm_cross = nn.LayerNorm(d_model)
        self.norm_ffn = nn.LayerNorm(d_model)

        # 4) Dropout stosowany na wyjściu każdej podwarstwy przed residual connection
        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def generate_causal_mask(seq_len):
        """
        Tworzy maskę przyczynową (causal mask), która blokuje "patrzenie w przyszłość".

        Cel:
        - token na pozycji i może widzieć tylko pozycje <= i
        - pozycje > i mają być zablokowane

        W tej wersji zwracamy maskę addytywną:
        - 0      -> pozycja dozwolona
        - -inf   -> pozycja zablokowana

        Kształt wyjścia:
            [seq_len, seq_len]
        """
        # TODO (student)
        # 1) Utwórz macierz jedynek [seq_len, seq_len]
        # 2) Weź część górnotrójkątną powyżej przekątnej (diagonal=1),
        #    aby zaznaczyć "przyszłe" pozycje
        #    Użyj: torch.triu(torch.ones(...), diagonal=1)
        mask = torch.triu(torch.ones(seq_len, seq_len), diagonal=1)

        # 3) Zamień jedynki (pozycje zakazane) na -inf
        #    Użyj: masked_fill(mask == 1, float('-inf'))
        #
        # Pozycje dozwolone pozostaną 0.
        return mask.masked_fill(mask == 1, float('-inf'))

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        """
        Wejście:
            x:          [batch, tgt_seq_len, d_model]       - stan dekodera
            enc_output: [batch, src_seq_len, d_model]       - wyjście enkodera
            src_mask:   maska dla cross-attention (opcjonalna)
            tgt_mask:   maska paddingu targetu (opcjonalna)

        Wyjście:
            [batch, tgt_seq_len, d_model]
        """
        batch_size, seq_len, _ = x.size()

        # TODO (student) — przygotowanie maski dla self-attention dekodera
        # 1) Utwórz maskę przyczynową dla długości sekwencji docelowej
        #    i przenieś ją na to samo urządzenie co x (CPU/GPU)
        causal_mask = self.generate_causal_mask(seq_len).to(x.device).unsqueeze(0).unsqueeze(0)

        # 2) Połącz maskę przyczynową z maską paddingu targetu (jeśli istnieje)
        #    W tej wersji zakładamy maski addytywne (0 / -inf), więc można je dodać.
        #    Jeśli używasz masek bool (0/1), łączenie robi się inaczej (np. AND / masked_fill).
        if tgt_mask is not None:
            full_tgt_mask = tgt_mask + causal_mask
        else:
            full_tgt_mask = causal_mask

        # TODO podwarstwy dekodera

        # PODWARSTWA 1: Masked Self-Attention
        # Q, K, V pochodzą z x (czyli z dekodera)
        # Używamy full_tgt_mask, żeby:
        # - nie patrzeć w przyszłość (causal mask)
        # - opcjonalnie ignorować padding targetu
        self_attn_out = self.self_attn(x, x, x, (full_tgt_mask == 0).to(x.dtype))

        # Residual + Dropout + LayerNorm
        # Schemat: x <- LayerNorm(x + Dropout(self_attn_out))
        x = self.norm_self(x + self.dropout(self_attn_out))

        # PODWARSTWA 2: Cross-Attention (Encoder-Decoder Attention)
        # Q pochodzi z dekodera (aktualne x)
        # K i V pochodzą z enkodera (enc_output)
        # Używamy src_mask, aby np. ukryć padding po stronie źródła
        cross_attn_out = self.cross_attn(x, enc_output, enc_output, src_mask)

        # Residual + Dropout + LayerNorm
        x = self.norm_cross(x + self.dropout(cross_attn_out))

        # PODWARSTWA 3: Feed-Forward Network
        # FFN działa na każdej pozycji sekwencji niezależnie
        ffn_out = self.ffn_block(x)

        # Residual + Dropout + LayerNorm
        x = self.norm_ffn(x + self.dropout(ffn_out))

        return x

In [8]:
net = TransformerDecoderLayer( d_model = 512, d_ff =2048, num_heads=8, dropout=0.1)
print(net)

TransformerDecoderLayer(
  (self_attn): MultiHeadAttentionModule(
    (q_linear): Linear(in_features=512, out_features=512, bias=True)
    (k_linear): Linear(in_features=512, out_features=512, bias=True)
    (v_linear): Linear(in_features=512, out_features=512, bias=True)
    (out_projection): Linear(in_features=512, out_features=512, bias=True)
    (attention_dropout): Dropout(p=0.1, inplace=False)
  )
  (cross_attn): MultiHeadAttentionModule(
    (q_linear): Linear(in_features=512, out_features=512, bias=True)
    (k_linear): Linear(in_features=512, out_features=512, bias=True)
    (v_linear): Linear(in_features=512, out_features=512, bias=True)
    (out_projection): Linear(in_features=512, out_features=512, bias=True)
    (attention_dropout): Dropout(p=0.1, inplace=False)
  )
  (ffn_block): PositionWiseFFN(
    (w_expand): Linear(in_features=512, out_features=2048, bias=True)
    (w_shrink): Linear(in_features=2048, out_features=512, bias=True)
    (dropout): Dropout(p=0.1, inplace=

### Zadanie 6: Budowa Stosów Enkodera i Dekodera

W tym zadaniu zaimplementujecie finalne struktury zarządzające wieloma warstwami obliczeniowymi. Kluczowym wyzwaniem jest tutaj poprawne przekazywanie danych: wynik pracy ostatniej warstwy enkodera musi trafić do każdej warstwy dekodera jako źródło informacji dla mechanizmu cross-attention.

Kluczowe założenia (Sekcja 3.1):
- **Liczba warstw ($N$)** - Zarówno enkoder, jak i dekoder składają się ze stosu $N=6$ identycznych warstw.+1
- **Przepływ danyc** Każda podwarstwa w całym stosie produkuje wyjście o wymiarze $d_{model} = 512$, co pozwala na zachowanie ciągłości połączeń rezydualnych wewnątrz całego stosu.
- **Mechanizm Attention** W warstwach dekodera zapytania (Queries) pochodzą z poprzedniej warstwy dekodera, natomiast klucze (Keys) i wartości (Values) pochodzą bezpośrednio z końcowego wyjścia stosu enkodera.

In [9]:
import copy
import torch
import torch.nn as nn

class TransformerStack(nn.Module):
    """
    Uniwersalny stos N warstw (np. enkodera albo dekodera).

    Idea:
    - dostajemy pojedynczy blok (warstwę),
    - tworzymy N niezależnych kopii,
    - przepuszczamy dane kolejno przez każdą warstwę,
    - na końcu opcjonalnie stosujemy LayerNorm.
    """
    def __init__(self, layer_block, num_layers):
        super().__init__()

        # TODO (student) — przechowywanie warstw
        # 1) Użyj nn.ModuleList(...), aby PyTorch poprawnie rejestrował warstwy
        #    (dzięki temu ich parametry będą widoczne dla optymalizatora i .to(device)).
        #
        # 2) WAŻNE: każda warstwa powinna być osobną kopią (osobne wagi),
        #    więc nie używaj samego: [layer_block for _ in range(num_layers)]
        #    bo to doda TEN SAM obiekt wiele razy.
        #
        #    Użyj: copy.deepcopy(layer_block)
        self.layers = nn.ModuleList([copy.deepcopy(layer_block) for _ in range(num_layers)])

        # 3) (Opcjonalnie) końcowa normalizacja po całym stosie.
        #    Zakładamy, że blok ma atrybut d_model.
        #    Użyj: nn.LayerNorm(layer_block.d_model)
        self.final_norm = nn.LayerNorm(layer_block.d_model)

    def forward(self, x, *args, **kwargs):
        """
        Sekwencyjne przetwarzanie:
        x -> warstwa1 -> warstwa2 -> ... -> warstwaN -> final_norm

        *args i **kwargs pozwalają przekazywać dodatkowe argumenty
        (np. maski, encoder output itp.) bez przepisywania tej klasy
        osobno dla enkodera i dekodera.
        """
        # TODO (student)
        # Przejdź po wszystkich warstwach w self.layers i za każdym razem:
        # x = layer(x, *args, **kwargs)
        for layer in self.layers:
            x = layer(x, *args, **kwargs)

        # Zastosuj końcową normalizację i zwróć wynik
        return self.final_norm(x)


class FullTransformerEncoder(nn.Module):
    """
    Pełny enkoder:
    1) embedding + pozycje
    2) stos N bloków enkodera
    """
    def __init__(self, vocab_size, d_model, d_ff, num_heads, num_layers, dropout=0.1):
        super().__init__()

        # TODO (student)
        # 1) Moduł wejściowy (embedding + pozycje + dropout)
        #    Użyj wcześniej przygotowanego TransformerInput
        self.input_module = TransformerInput(vocab_size, d_model, d_model, dropout)

        # 2) Zbuduj pojedynczy blok enkodera
        encoder_layer = EncoderBlock(d_model, num_heads, d_ff, dropout)

        # 3) Utwórz stos N warstw enkodera (z niezależnymi kopiami wag)
        #    Użyj klasy TransformerStack
        self.stack = TransformerStack(encoder_layer, num_layers)

    def forward(self, x, mask=None):
        """
        Wejście:
            x: [batch, src_seq_len] (tokeny)
            mask: maska paddingu dla enkodera (opcjonalna)

        Wyjście:
            [batch, src_seq_len, d_model]
        """
        # Krok 1) Zamień tokeny na reprezentacje wektorowe + dodaj pozycje
        x = self.input_module(x)

        # Krok 2) Przepuść przez stos bloków enkodera
        # Uwaga: EncoderBlock przyjmuje argument padding_mask, więc przekazujemy go nazwą
        x = self.stack(x, padding_mask=mask)

        return x


class FullTransformerDecoder(nn.Module):
    """
    Pełny dekoder:
    1) embedding + pozycje
    2) stos N bloków dekodera
    """
    def __init__(self, vocab_size, d_model, d_ff, num_heads, num_layers, dropout=0.1):
        super().__init__()

        # TODO (student)
        # 1) Moduł wejściowy dla tokenów docelowych (tgt)
        self.input_module = TransformerInput(vocab_size, d_model, d_model, dropout)

        # 2) Utwórz pojedynczy blok dekodera
        #    (zakładamy, że masz klasę TransformerDecoderLayer)
        decoder_layer = TransformerDecoderLayer(d_model, d_ff, num_heads, dropout)

        # 3) Zbuduj stos N bloków dekodera
        self.stack = TransformerStack(decoder_layer, num_layers)

    def forward(self, x, enc_output, src_mask=None, tgt_mask=None):
        """
        Wejście:
            x: [batch, tgt_seq_len]              - tokeny wejściowe dekodera
            enc_output: [batch, src_seq_len, d_model] - wyjście enkodera
            src_mask: maska dla uwagi krzyżowej (encoder-decoder attention)
            tgt_mask: maska dla self-attention dekodera (padding +/lub causal mask)

        Wyjście:
            [batch, tgt_seq_len, d_model]
        """
        # Krok 1) Embedding + pozycje dla sekwencji docelowej
        x = self.input_module(x)

        # Krok 2) Przepuszczenie przez wszystkie warstwy dekodera
        # Każda warstwa dostaje:
        # - aktualny stan dekodera x
        # - wyjście enkodera enc_output
        # - maski src/tgt
        #
        # Zakładamy, że TransformerDecoderLayer ma forward zgodny z:
        # layer(x, enc_output, src_mask=None, tgt_mask=None)
        x = self.stack(x, enc_output, src_mask=src_mask, tgt_mask=tgt_mask)

        return x

### Zadanie 7: Pełna Architektura Transformer

W tym zadaniu zbudujecie finalną klasę, która zarządza przepływem danych między koderem a dekoderem. Zgodnie z **Sekcją 3** publikacji, model ten działa w sposób auto-regresywny generuje jeden symbol na raz, wykorzystując poprzednio stworzone znaki jako dodatkowe wejście.

**Kluczowe aspekty integracji (Sekcje 3.1 - 3.4):**

* **Współdzielenie wag -** Często stosowaną praktyką jest używanie tej samej macierzy wag dla osadzeń wejściowych, wyjściowych oraz warstwy liniowej przed funkcją softmax.

* **Skalowanie -** Pamiętajcie o pomnożeniu wag w warstwach osadzeń przez .

* **Przesunięcie wyjścia -** Podczas treningu (teacher forcing), wejście dekodera jest przesunięte o jedną pozycję w prawo względem celu, co zapewnia, że przewidywanie dla pozycji  opiera się tylko na pozycjach mniejszych niż .

* **Stabilność numeryczna -** Zamiast czystej funkcji softmax, na wyjściu stosuje się często `log_softmax`, co poprawia stabilność obliczeń podczas wyliczania funkcji straty.


In [10]:
import torch
import torch.nn as nn

class TransformerModel(nn.Module):
    """
    Kompletna architektura Transformer:
    - encoder (przetwarza sekwencję źródłową)
    - decoder (generuje reprezentacje sekwencji docelowej)
    - warstwa wyjściowa (projekcja do słownika)
    """
    def __init__(self, encoder_stack, decoder_stack, target_vocab_size):
        super().__init__()
        self.encoder = encoder_stack
        self.decoder = decoder_stack

        # TODO (student) — głowica wyjściowa
        # 1) Warstwa liniowa mapująca z d_model -> target_vocab_size
        #    Dzięki temu dla każdej pozycji sekwencji dostajemy logity dla wszystkich tokenów.
        #    Użyj: nn.Linear(..., target_vocab_size)
        #
        # Uwaga: decoder_stack musi mieć atrybut d_model (np. self.d_model = d_model w klasie dekodera)
        decoder_d_model = decoder_stack.d_model if hasattr(decoder_stack, "d_model") else decoder_stack.input_module.d_model
        self.final_projection = nn.Linear(decoder_d_model, target_vocab_size)

        # 2) LogSoftmax po ostatnim wymiarze (wymiar słownika)
        #    Zwracamy log-prawdopodobieństwa.
        #    Użyj: nn.LogSoftmax(dim=-1)
        #
        # Uwaga praktyczna:
        # - jeśli używasz nn.NLLLoss -> log_softmax jest OK
        # - jeśli używasz nn.CrossEntropyLoss -> zwykle zwraca się surowe logity (bez softmax/log_softmax)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def create_start_token_shift(self, target_tokens):
        """
        Przygotowuje wejście dekodera (teacher forcing):
        przesuwa target o 1 w prawo i wstawia token startowy na początek.

        Przykład:
        target        = [t1, t2, t3, t4]
        shifted_input = [<SOS>, t1, t2, t3]
        """
        # TODO (student)
        # 1) Pobierz rozmiar batcha
        #    Użyj: target_tokens.size(0)
        batch_size = target_tokens.size(0)

        # 2) Utwórz kolumnę tokenów startowych o kształcie [batch_size, 1]
        #    Użyj: torch.zeros(...), jeśli indeks startowego tokenu = 0
        #    Ważne: ustaw dtype i device takie same jak w target_tokens
        start_tokens = torch.zeros(batch_size, 1, dtype=target_tokens.dtype, device=target_tokens.device)

        # 3) Sklej token startowy z targetem bez ostatniego elementu
        #    target_tokens[:, :-1] usuwa ostatni token z każdej sekwencji
        #    Użyj: torch.cat([...], dim=1)
        #
        # Wynik ma ten sam kształt co target_tokens: [batch_size, tgt_seq_len]
        return torch.cat([start_tokens, target_tokens[:, :-1]], dim=1)

    def forward(self, source, target, src_mask=None, tgt_mask=None):
        """
        Główny przepływ danych przez model.

        Wejście:
            source: [batch_size, src_seq_len]  - tokeny źródłowe
            target: [batch_size, tgt_seq_len]  - tokeny docelowe (prawdziwe, do teacher forcing)
            src_mask: maska dla enkodera / cross-attention (opcjonalna)
            tgt_mask: maska dla self-attention dekodera (np. causal mask + padding, opcjonalna)

        Wyjście:
            log_probs: [batch_size, tgt_seq_len, target_vocab_size]
        """
        # 1) Przygotowanie wejścia dekodera: przesunięcie targetu w prawo
        shifted_target = self.create_start_token_shift(target)

        # TODO (student) — przepływ przez encoder i decoder

        # 2) Encoder: kodowanie sekwencji źródłowej
        #    Wynik to "pamięć" dla dekodera (encoder memory)
        #    Kształt: [batch_size, src_seq_len, d_model]
        encoded_memory = self.encoder(source, mask=src_mask)

        # 3) Decoder:
        #    - wejście: shifted_target
        #    - pamięć z enkodera: encoded_memory
        #    - maski: src_mask, tgt_mask
        #
        #    Wynik: [batch_size, tgt_seq_len, d_model]
        decoded_sequence = self.decoder(shifted_target, encoded_memory, src_mask=src_mask, tgt_mask=tgt_mask)

        # 4) Projekcja do rozmiaru słownika (logity dla każdego tokenu)
        #    Kształt: [batch_size, tgt_seq_len, target_vocab_size]
        logits = self.final_projection(decoded_sequence)

        # 5) Zamiana logitów na log-prawdopodobieństwa
        #    (jeśli trenujesz z NLLLoss)
        return self.log_softmax(logits)

## Cz. II
1. Przygotowanie danych.
2. Strategia uczenia.
3. Mechanizm wnioskowania.
4. Ewaluacja i wizualizacja.

In [11]:
import random
import math
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

Jasne — poniżej masz z tego zrobione **zadanie dla studentów**, w stylu spójnym z wcześniejszymi poleceniami.

---

### Zadanie 1. Konfiguracja eksperymentu dla modelu Transformer

W tej części zadania należy przygotować kompletną konfigurację środowiska oraz hiperparametrów potrzebnych do uruchomienia i trenowania własnej architektury Transformer na prostym zadaniu sekwencyjnym. Celem tego etapu nie jest jeszcze samo uczenie modelu, ale poprawne zdefiniowanie wszystkich ustawień, od których zależy stabilność eksperymentu, powtarzalność wyników oraz poprawność późniejszego pipeline’u treningowego.

W praktyce oznacza to przygotowanie:

* urządzenia obliczeniowego,
* ziaren losowości,
* tokenów specjalnych,
* zakresu symboli używanych w zadaniu,
* długości sekwencji,
* podstawowych hiperparametrów modelu,
* oraz hiperparametrów procesu uczenia.

#### Kluczowe założenia

* **Urządzenie obliczeniowe** – model powinien automatycznie wykrywać, czy dostępne jest GPU, i w zależności od tego uruchamiać się na `cuda` albo `cpu`.
* **Powtarzalność eksperymentu** – należy ustawić ziarna losowości dla bibliotek używanych podczas generowania danych i trenowania modelu.
* **Tokeny specjalne** – trzeba zdefiniować indeksy dla tokenów `PAD`, `SOS` oraz `EOS`, które będą wykorzystywane podczas przygotowania danych wejściowych i docelowych.
* **Słownik symboli** – należy przyjąć, że zwykłe liczby od `1` do `20` są mapowane na indeksy większe od tokenów specjalnych, tak aby nie mieszać danych właściwych z tokenami technicznymi.
* **Długość sekwencji** – trzeba określić minimalną i maksymalną długość generowanych sekwencji wejściowych.
* **Hiperparametry modelu** – należy ustawić główny wymiar modelu `d_model`, szerokość warstwy feed-forward `d_ff`, liczbę głowic attention, liczbę warstw oraz współczynnik dropout.
* **Hiperparametry treningu** – należy zdefiniować wielkość batcha, liczbę epok oraz współczynnik uczenia.

#### Wskazówki do napisania kodu

* Do wyboru urządzenia użyjcie mechanizmu:
  `torch.device("cuda" if torch.cuda.is_available() else "cpu")`
* Dla powtarzalności ustawcie ziarna losowości przy pomocy:
  `random.seed(...)` oraz `torch.manual_seed(...)`
* Przyjmijcie następujące tokeny specjalne:

  * `PAD_IDX = 0`
  * `SOS_IDX = 1`
  * `EOS_IDX = 2`
* Załóżcie, że liczby od `1` do `20` są kodowane jako kolejne tokeny po tokenach specjalnych, co daje łącznie słownik o rozmiarze `23`
* Ograniczcie długość sekwencji do przedziału od `3` do `8`
* Ustawcie lekką konfigurację modelu odpowiednią do eksperymentów w Google Colab:

  * `d_model = 128`
  * `d_ff = 256`
  * `num_heads = 4`
  * `num_layers = 2`
  * `dropout = 0.05`
* Ustalcie parametry treningu:

  * `batch_size = 32`
  * `num_epochs = 20`
  * `learning_rate = 1e-3`

In [12]:
# TODO:
# 1) Wybierz urządzenie obliczeniowe:
#    - jeśli dostępne jest GPU, użyj "cuda"
#    - w przeciwnym wypadku użyj "cpu"
#    Użyj: torch.device(...)
device =

print("Urządzenie:", device)

# TODO:
# 2) Ustaw ziarna losowości dla powtarzalności eksperymentu
#    Użyj: random.seed(...), torch.manual_seed(...)
random.seed(...)
torch.manual_seed(...)

# TODO:
# 3) Zdefiniuj tokeny specjalne:
#    - PAD_IDX jako 0
#    - SOS_IDX jako 1
#    - EOS_IDX jako 2
PAD_IDX =
SOS_IDX =
EOS_IDX =

# TODO:
# 4) Zdefiniuj zakres zwykłych symboli:
#    liczby od 1 do 20
MIN_NUMBER =
MAX_NUMBER =

# TODO:
# 5) Ustal rozmiar słownika:
#    3 tokeny specjalne + 20 zwykłych symboli
VOCAB_SIZE =

# TODO:
# 6) Ustal minimalną i maksymalną długość sekwencji
MIN_LEN =
MAX_LEN =

# TODO:
# 7) Ustaw podstawowe hiperparametry modelu
#    Użyj:
#    - d_model = 128
#    - d_ff = 256
#    - num_heads = 4
#    - num_layers = 2
#    - dropout = 0.05
d_model =
d_ff =
num_heads =
num_layers =
dropout =

# TODO:
# 8) Ustaw hiperparametry treningu
#    Użyj:
#    - batch_size = 32
#    - num_epochs = 20
#    - learning_rate = 1e-3
batch_size =
num_epochs =
learning_rate =

SyntaxError: invalid syntax (2592110106.py, line 6)

## Zadanie 2. Przygotowanie syntetycznego zbioru danych dla modelu Transformer

W tej części należy zaimplementować prosty generator danych, który posłuży do trenowania modelu Transformer w zadaniu odwracania sekwencji. Celem jest przygotowanie par danych wejściowych i wyjściowych w taki sposób, aby model otrzymywał sekwencję w zwykłej kolejności, a następnie uczył się zwracać tę samą sekwencję od końca.

Zadanie opiera się na sztucznie generowanych przykładach, dzięki czemu nie ma potrzeby korzystania z gotowych korpusów tekstowych ani zewnętrznych plików. Każdy przykład ma reprezentować prostą parę typu **source-target**, gdzie:

* `source` to sekwencja wejściowa,
* `target` to odpowiadająca jej sekwencja docelowa.

Najpierw należy przygotować funkcję odpowiedzialną za zamianę zwykłej liczby na token używany przez model. W eksperymencie przyjęto, że liczby od **1 do 20** nie będą używane bezpośrednio jako indeksy, ponieważ najniższe indeksy są już zarezerwowane dla tokenów specjalnych. Oznacza to, że właściwe symbole muszą zostać przesunięte tak, aby liczba `1` odpowiadała pierwszemu zwykłemu tokenowi, a liczba `20` ostatniemu. Następnie trzeba przygotować funkcję odwrotną, która pozwoli przejść z tokenu z powrotem do zwykłej liczby. Jest to potrzebne po to, aby później można było łatwo interpretować wygenerowane dane i predykcje modelu.

Kolejnym krokiem jest implementacja funkcji generującej pojedynczy przykład. W każdym wywołaniu powinna być losowana długość sekwencji. Długość ta ma mieścić się w ustalonym wcześniej przedziale, czyli wynosić mniej więcej od **3 do 8 elementów**. Następnie należy wylosować samą zawartość sekwencji, korzystając z liczb całkowitych z zakresu od **1 do 20**. Wylosowane liczby stanowią bazę przykładu, z której zostaną utworzone zarówno dane wejściowe, jak i dane docelowe.

Sekwencja `source` ma zachowywać oryginalną kolejność elementów. Oznacza to, że wygenerowane liczby należy tylko zamienić na odpowiadające im tokeny modelu, ale nie wolno zmieniać ich kolejności. Z kolei sekwencja `target` ma być zbudowana z tych samych elementów, ale zapisanych od końca. Jest to sedno całego zadania, ponieważ model ma się nauczyć właśnie tej zależności. Na końcu sekwencji docelowej należy dodatkowo dopisać token końca sekwencji `EOS_IDX`, aby model w dalszych etapach treningu mógł nauczyć się również momentu zakończenia generowania.

Po przygotowaniu pojedynczego przykładu trzeba zaimplementować funkcję budującą cały zbiór danych. Jej zadaniem będzie wielokrotne wywołanie generatora pojedynczych przykładów i zapisanie wyników w jednej liście. Każdy element tej listy powinien być parą `(source, target)`. Następnie należy wygenerować dwa osobne zbiory:

* zbiór treningowy zawierający około **20000 przykładów**,
* zbiór testowy zawierający około **2000 przykładów**.

Takie rozmiary są wystarczające, aby model miał z czego się uczyć, a jednocześnie cały eksperyment pozostaje lekki i możliwy do wykonania w środowisku takim jak Google Colab. Na końcu należy wypisać jeden przykładowy element zbioru treningowego, aby upewnić się, że dane zostały przygotowane poprawnie i że zależność między wejściem a wyjściem rzeczywiście odpowiada odwracaniu sekwencji.

Z punktu widzenia teorii ten etap odpowiada przygotowaniu danych do zadania typu **sequence-to-sequence**. Transformer nie uczy się tutaj jeszcze znaczenia językowego, ale relacji pomiędzy dwiema sekwencjami.


In [ ]:
import random

def number_to_token(x):
    # TODO:
    # liczba 1..20 -> token 3..22
    # Użyj przesunięcia o 2
    return

def token_to_number(tok):
    # TODO:
    # token 3..22 -> liczba 1..20
    # Odwróć wcześniejsze mapowanie
    return

def generate_example():
    # TODO:

    # 1) Wylosuj długość sekwencji z zakresu MIN_LEN..MAX_LEN
    #    Użyj: random.randint(...)
    seq_len =

    # 2) Wygeneruj losową sekwencję liczb z zakresu MIN_NUMBER..MAX_NUMBER
    #    Użyj list comprehension i random.randint(...)
    numbers =

    # 3) Zbuduj source:
    #    - zachowaj zwykłą kolejność
    #    - każdą liczbę zamień na token funkcją number_to_token(...)
    source =

    # 4) Zbuduj target:
    #    - odwróć kolejność sekwencji
    #    - zamień liczby na tokeny
    #    - na końcu dodaj token EOS_IDX
    target =

    return source, target

def build_dataset(num_samples):
    # TODO:
    # Utwórz listę num_samples przykładów przy użyciu generate_example()
    data =
    return data

# TODO:
# Wygeneruj:
# - zbiór treningowy o rozmiarze 20000
# - zbiór testowy o rozmiarze 2000
train_data =
test_data =

print("Przykład treningowy:")
print(train_data[0])

## Zadanie 3. Przygotowanie `Dataset` i `DataLoader` dla modelu Transformer

W tej części należy przygotować mechanizm ładowania danych do modelu Transformer. Wcześniej wygenerowane przykłady `source-target` są jeszcze zapisane jako zwykłe listy, natomiast model oczekuje danych w postaci tensorów PyTorch, pogrupowanych w batchach. Trzeba więc przygotować:

* klasę `Dataset`,
* funkcję `collate_fn`,
* oraz `DataLoader` dla zbioru treningowego i testowego.

Klasa datasetu powinna zwracać pojedynczy przykład w postaci dwóch tensorów typu `torch.long`: sekwencji wejściowej `source` oraz sekwencji docelowej `target`. Jest to konieczne, ponieważ warstwy embeddingowe operują właśnie na indeksach całkowitych.

Ponieważ sekwencje mają różne długości, nie można ich od razu połączyć w jeden batch. Z tego powodu należy zaimplementować funkcję `collate_fn`, która:

* wyznacza maksymalną długość sekwencji w batchu,
* dopełnia krótsze sekwencje tokenem `PAD_IDX`,
* a następnie łączy wszystko w jeden tensor.

Na końcu trzeba utworzyć dwa obiekty `Dataset`:

* `train_dataset`,
* `test_dataset`,

a następnie dwa obiekty `DataLoader`, które będą wykorzystywane podczas treningu i testowania modelu. W przypadku zbioru treningowego należy włączyć mieszanie danych, natomiast w przypadku zbioru testowego nie jest to potrzebne.

Z punktu widzenia teorii jest to etap przygotowania danych do przetwarzania wsadowego. Padding jest tutaj konieczny, ponieważ Transformer pracuje na tensorach o regularnych kształtach, a sekwencje w naturalny sposób mają zmienną długość. Dzięki temu dane będą gotowe do dalszego etapu, czyli trenowania modelu.

In [ ]:
class ReverseSequenceDataset(Dataset):
    def __init__(self, data):
        # TODO:
        # zapisz dane w obiekcie klasy
        self.data =

    def __len__(self):
        # TODO:
        # zwróć liczbę przykładów w zbiorze
        return

    def __getitem__(self, idx):
        # TODO:
        # pobierz przykład o indeksie idx
        # przykład składa się z pary: (source, target)
        source, target =

        # TODO:
        # zamień source i target na tensory typu torch.long
        return


def collate_fn(batch):
    # batch = lista par: (source_tensor, target_tensor)

    # TODO:
    # rozdziel batch na osobne listy source i target
    sources, targets =

    # TODO:
    # oblicz maksymalną długość source i target w batchu
    max_src_len =
    max_tgt_len =

    padded_sources = []
    padded_targets = []

    for src, tgt in zip(sources, targets):
        # TODO:
        # oblicz, ile paddingu trzeba dodać do source
        src_pad_len =

        # TODO:
        # uzupełnij source tokenem PAD_IDX do długości max_src_len
        padded_src =

        padded_sources.append(padded_src)

        # TODO:
        # oblicz, ile paddingu trzeba dodać do target
        tgt_pad_len =

        # TODO:
        # uzupełnij target tokenem PAD_IDX do długości max_tgt_len
        padded_tgt =

        padded_targets.append(padded_tgt)

    # TODO:
    # połącz listy padded_sources i padded_targets w tensory batchowe
    padded_sources =
    padded_targets =

    return padded_sources, padded_targets


# TODO:
# utwórz obiekty datasetu dla train_data i test_data
train_dataset =
test_dataset =

# TODO:
# utwórz DataLoader dla zbioru treningowego i testowego
# dla train_loader ustaw shuffle=True
# dla test_loader ustaw shuffle=False
# użyj collate_fn=collate_fn
train_loader =
test_loader =

## Zadanie 4. Przygotowanie maski źródłowej dla mechanizmu attention

W tej części należy przygotować funkcję tworzącą maskę źródłową dla sekwencji wejściowej. Jej zadaniem jest wskazanie, które pozycje w tensorze wejściowym odpowiadają prawdziwym tokenom, a które są jedynie paddingiem dodanym podczas wyrównywania długości sekwencji w batchu.

Maska ta jest potrzebna, ponieważ mechanizm attention nie powinien uwzględniać pozycji zawierających `PAD`. Gdyby tego nie zrobić, model mógłby traktować sztucznie dodane tokeny jako część rzeczywistych danych wejściowych, co pogarszałoby jakość uczenia i wprowadzało niepotrzebny szum.

Funkcja powinna przyjmować tensor `source_tokens` o kształcie:

* `[batch, src_seq_len]`

i zwracać maskę o kształcie:

* `[batch, 1, 1, src_seq_len]`

Wartość `1` powinna oznaczać prawdziwy token, natomiast `0` pozycję paddingową. Taki kształt jest zgodny z mechanizmem broadcastingu stosowanym w attention i pozwala poprawnie rozszerzyć maskę na wszystkie głowice oraz wszystkie pozycje zapytań.

In [ ]:
def create_src_mask(source_tokens):
    # source_tokens: [batch, src_seq_len]

    # TODO:
    # utwórz maskę, w której:
    # - 1 oznacza prawdziwy token
    # - 0 oznacza token PAD
    mask =

    # TODO:
    # dopasuj kształt maski do attention:
    # [batch, 1, 1, src_seq_len]
    mask =

    return mask

## Zadanie 5. Budowa pełnego modelu Transformer oraz przygotowanie elementów treningu

Po przygotowaniu wszystkich wcześniejszych komponentów można przejść do złożenia ich w jedną, spójną architekturę. W tym etapie należy utworzyć pełny enkoder, pełny dekoder, a następnie połączyć je w końcowy model Transformer odpowiedzialny za przetwarzanie sekwencji wejściowej i generowanie sekwencji wyjściowej.

Oprócz samego modelu trzeba również przygotować elementy niezbędne do treningu:

* optymalizator,
* funkcję straty.

Enkoder i dekoder powinny zostać utworzone przy użyciu tych samych hiperparametrów, takich jak rozmiar słownika, wymiar modelu, liczba głowic, liczba warstw oraz dropout. Następnie oba moduły należy przekazać do klasy `TransformerModel`, a gotowy model przenieść na odpowiednie urządzenie obliczeniowe.

Na końcu należy utworzyć optymalizator `Adam` oraz funkcję straty `NLLLoss` z parametrem `ignore_index=PAD_IDX`, aby pozycje odpowiadające paddingowi nie wpływały na obliczanie błędu.


In [ ]:
# TODO:
# utwórz pełny enkoder transformera
encoder = FullTransformerEncoder(
    vocab_size=,
    d_model=,
    d_ff=,
    num_heads=,
    num_layers=,
    dropout=
)

# TODO:
# utwórz pełny dekoder transformera
decoder = FullTransformerDecoder(
    vocab_size=,
    d_model=,
    d_ff=,
    num_heads=,
    num_layers=,
    dropout=
)

# TODO:
# połącz enkoder i dekoder w pełny model Transformer
# oraz przenieś model na odpowiednie urządzenie
model = TransformerModel(
    encoder_stack=,
    decoder_stack=,
    target_vocab_size=
).to(...)

# TODO:
# utwórz optymalizator Adam dla parametrów modelu
optimizer =

# TODO:
# utwórz funkcję straty NLLLoss
# padding ma być ignorowany przy liczeniu loss
loss_fn =

print("Model utworzony poprawnie.")

## Zadanie 6. Implementacja jednej epoki treningowej modelu

Po zbudowaniu modelu oraz przygotowaniu danych można przejść do implementacji pojedynczej epoki treningowej. Celem tego etapu jest wykonanie pełnego przebiegu po wszystkich batchach ze zbioru treningowego, obliczenie funkcji straty, wykonanie propagacji wstecznej oraz aktualizacja parametrów modelu.

W każdej iteracji batch danych powinien zostać przeniesiony na odpowiednie urządzenie obliczeniowe. Następnie należy utworzyć maskę źródłową dla sekwencji wejściowej, wyzerować gradienty, wykonać przejście w przód przez model i obliczyć funkcję straty. Po wyznaczeniu błędu trzeba wykonać propagację wsteczną oraz krok optymalizatora.

Na końcu funkcja powinna zwrócić średnią wartość straty z całej epoki. Jest to podstawowa miara pozwalająca obserwować, czy model uczy się poprawnie i czy wartość błędu maleje w kolejnych epokach.


In [ ]:
def train_one_epoch(model, dataloader, optimizer, loss_fn, device):
    # TODO:
    # ustaw model w tryb treningowy
    model.

    total_loss = 0.0

    for source, target in dataloader:
        # TODO:
        # przenieś source i target na odpowiednie urządzenie
        source =
        target =

        # TODO:
        # utwórz maskę źródłową i przenieś ją na urządzenie
        src_mask =

        # TODO:
        # wyzeruj gradienty optymalizatora
        optimizer.

        # TODO:
        # wykonaj przejście w przód przez model
        # log_probs: [batch, tgt_seq_len, vocab_size]
        log_probs =

        # TODO:
        # oblicz funkcję straty
        # NLLLoss oczekuje:
        # input:  [batch, vocab_size, tgt_seq_len]
        # target: [batch, tgt_seq_len]
        loss =

        # TODO:
        # wykonaj propagację wsteczną
        loss.

        # TODO:
        # wykonaj krok optymalizatora
        optimizer.

        # TODO:
        # dodaj wartość straty do sumy
        total_loss +=

    # TODO:
    # zwróć średnią stratę z całej epoki
    return

## Zadanie 7. Implementacja procedury ewaluacji modelu

Po przygotowaniu funkcji treningowej należy zaimplementować procedurę ewaluacji modelu. Jej celem jest obliczenie średniej wartości funkcji straty na zbiorze walidacyjnym lub testowym, bez aktualizowania parametrów modelu.

W odróżnieniu od treningu model powinien zostać przełączony w tryb ewaluacyjny, co wyłącza mechanizmy takie jak dropout. Dodatkowo obliczenia należy wykonać w bloku `torch.no_grad()`, aby nie śledzić gradientów i nie zużywać niepotrzebnie pamięci. Dla każdego batcha trzeba przenieść dane na odpowiednie urządzenie, utworzyć maskę źródłową, wykonać przejście w przód przez model i obliczyć funkcję straty. Na końcu funkcja powinna zwrócić średnią wartość straty z całego zbioru. Pozwala to ocenić, jak model radzi sobie na danych, których nie wykorzystuje bezpośrednio do aktualizacji wag.

In [ ]:
def evaluate(model, dataloader, loss_fn, device):
    # TODO:
    # ustaw model w tryb ewaluacyjny
    model.

    total_loss = 0.0

    # TODO:
    # wyłącz śledzenie gradientów
    with torch.no_grad():
        for source, target in dataloader:
            # TODO:
            # przenieś source i target na odpowiednie urządzenie
            source =
            target =

            # TODO:
            # utwórz maskę źródłową i przenieś ją na urządzenie
            src_mask =

            # TODO:
            # wykonaj przejście w przód przez model
            log_probs =

            # TODO:
            # oblicz funkcję straty
            loss =

            # TODO:
            # dodaj wartość straty do sumy
            total_loss +=

    # TODO:
    # zwróć średnią stratę z całego zbioru
    return

## Zadanie 8. Implementacja głównej pętli treningowej

Po przygotowaniu funkcji odpowiedzialnych za trening jednej epoki oraz ewaluację modelu należy zbudować główną pętlę treningową. Jej zadaniem jest wielokrotne uruchamianie procesu uczenia i oceny modelu w kolejnych epokach oraz zapisywanie uzyskanych wyników.

W każdej epoce należy:

* obliczyć stratę treningową,
* obliczyć stratę testową,
* zapisać obie wartości do osobnych list,
* a następnie wypisać je w czytelnej postaci.


In [ ]:
# TODO:
# utwórz dwie puste listy:
# - train_losses
# - test_losses
train_losses =
test_losses =

# TODO:
# wykonaj pętlę po kolejnych epokach
for epoch in range(...):
    # TODO:
    # oblicz stratę treningową dla jednej epoki
    train_loss =

    # TODO:
    # oblicz stratę testową
    test_loss =

    # TODO:
    # zapisz obie wartości do odpowiednich list
    train_losses.
    test_losses.

    # TODO:
    # wypisz numer epoki oraz obie wartości straty
    print(...)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(train_losses, label="strata treningowa")
plt.plot(test_losses, label="strata testowa")
plt.xlabel("Epoka")
plt.ylabel("Loss")
plt.title("Przebieg uczenia modelu")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def greedy_decode(model, source_tokens, max_target_len=20):
    model.eval()

    source_tokens = source_tokens.to(device)
    src_mask = create_src_mask(source_tokens).to(device)

    with torch.no_grad():
        # kodowanie wejścia
        # TODO:
        # użyj enkodera modelu, aby uzyskać pamięć dla dekodera
        encoded_memory = ...

        # TODO:
        # uruchom dekoder na aktualnie wygenerowanej sekwencji
        generated = ...

        for _ in range(max_target_len):
            decoded = model.decoder(
                generated,
                encoded_memory,
                src_mask=src_mask,
                tgt_mask=None
            )

            logits = model.final_projection(decoded)
            log_probs = model.log_softmax(logits)

            next_token = log_probs[:, -1, :].argmax(dim=-1, keepdim=True)
            generated = torch.cat([generated, next_token], dim=1)

            if (next_token == EOS_IDX).all():
                break

    return generated

## Zadanie 10. Zamiana tokenów na zwykłe liczby

Model zwraca wyniki w postaci tokenów, dlatego przed analizą trzeba je zamienić z powrotem na zwykłe liczby. W tym celu należy przygotować dwie funkcje: jedną dla sekwencji wejściowej, a drugą dla sekwencji docelowej.

W sekwencji źródłowej należy pominąć token `PAD`, ponieważ jest on tylko technicznym wypełnieniem. W sekwencji docelowej trzeba dodatkowo pominąć `SOS` oraz zatrzymać odczyt po pojawieniu się `EOS`, ponieważ oznacza on koniec właściwej odpowiedzi.

Dzięki temu możliwe będzie czytelne porównywanie wejścia, oczekiwanego wyjścia i predykcji modelu.

In [ ]:
def decode_source_tokens(tokens):
    result = []
    for tok in tokens:
        tok = int(tok)
        if tok == PAD_IDX:
            continue
        # TODO:
        # dodaj zwykłą liczbę do wyniku
        result.append(...)
    return result

def decode_target_tokens(tokens):
    result = []
    for tok in tokens:
        # TODO:
        # zamień token na int
        tok =
        if tok in [PAD_IDX, SOS_IDX]:
            continue
        if tok == EOS_IDX:
            break
        result.append(token_to_number(tok))
    return result

## Zadanie 11. Wyświetlanie przykładowych predykcji modelu

Po wytrenowaniu modelu warto sprawdzić, jak wyglądają jego rzeczywiste odpowiedzi. W tym celu należy przygotować funkcję, która pobierze kilka przykładów z `DataLoadera`, wygeneruje predykcje przy pomocy dekodowania zachłannego, a następnie wypisze:

* sekwencję wejściową,
* oczekiwane wyjście,
* predykcję modelu.

In [ ]:
def show_predictions(model, dataloader, num_examples=5):
    # TODO:
    # ustaw model w tryb ewaluacyjny
    model.eval()

    # TODO:
    # pobierz jeden batch z dataloadera
    source_batch, target_batch =

    # TODO:
    # wybierz tylko pierwsze num_examples przykładów
    source_batch =
    target_batch =

    # TODO:
    # wygeneruj predykcje przy użyciu greedy_decode
    predicted =

    # TODO:
    # przenieś tensory na CPU
    source_batch =
    target_batch =
    predicted =

    for i in range(num_examples):
        # TODO:
        # zdekoduj source, target i predykcję do zwykłych liczb
        source_seq =
        target_seq =
        pred_seq =

        print(f"Przykład {i+1}")
        print("Wejście:     ", source_seq)
        print("Oczekiwane:  ", target_seq)
        print("Predykcja:   ", pred_seq)
        print("-" * 40)

# TODO:
# uruchom funkcję dla modelu i test_loadera
show_predictions(...)

## Zadanie 12. Obliczanie dokładności pełnej sekwencji

Aby ocenić model, należy obliczyć dokładność pełnej sekwencji. Wynik uznajemy za poprawny tylko wtedy, gdy cała przewidziana sekwencja jest identyczna z sekwencją oczekiwaną.

Uzupełnij funkcję tak, aby:

* wygenerować predykcje modelu,
* a następnie obliczyć i wypisać dokładność.


In [ ]:
def sequence_accuracy(model, dataloader, max_examples=200):
    model.eval()

    total = 0
    correct = 0

    with torch.no_grad():
        for source_batch, target_batch in dataloader:
            source_batch = source_batch.to(device)
            target_batch = target_batch.to(device)

            # TODO:
            # wygeneruj predykcje dla source_batch
            predicted =

            source_batch = source_batch.cpu()
            target_batch = target_batch.cpu()
            predicted = predicted.cpu()

            for i in range(source_batch.size(0)):
                true_seq = decode_target_tokens(target_batch[i])
                pred_seq = decode_target_tokens(predicted[i])

                if true_seq == pred_seq:
                    correct += 1
                total += 1

                if total >= max_examples:
                    return correct / total

    return correct / total

# TODO:
# oblicz dokładność i zapisz ją do zmiennej acc
acc =

print(f"Dokładność pełnej sekwencji: {acc:.4f}")